# 🧊 Iceberg Superpowers — Schema & Partition Evolution

Features that were **impossible** (or incredibly painful) in older systems like Hadoop/Hive, delivered as instant metadata operations in Iceberg.

| Feature | Hive/Hadoop 😰 | Iceberg 🧊 |
|---------|---------------|-------------|
| Add a column | Rewrite all Parquet files | Metadata-only, instant |
| Rename a column | Not supported (positional reads) | Metadata-only, instant |
| Change partitioning | Create new table + full rewrite + migrate | Metadata-only, old data untouched |

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Connect to Trino

In [1]:
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

✅ Connected to Trino


---
# Part 1 — Schema Evolution (Zero-Rewrite Magic)

In Hive, schemas are **coupled to the physical file layout**. Adding a column means rewriting every Parquet file. Renaming a column? Simply not possible — Hive uses positional column mapping, so column identity = column position.

Iceberg tracks columns by **unique IDs**, not positions or names. Every schema change is a metadata-only operation — zero data files are rewritten, regardless of table size.

## 1.1 — Create a Product Catalog Table

In [2]:
run_query("DROP TABLE IF EXISTS iceberg.bronze.products")

run_query("""
CREATE TABLE iceberg.bronze.products (
    product_id   BIGINT,
    name         VARCHAR,
    price        DOUBLE,
    category     VARCHAR
) WITH (format = 'PARQUET')
""")
print("✅ Table 'products' created")

✅ Table 'products' created


In [4]:
run_query("""
INSERT INTO iceberg.bronze.products VALUES
    (1, 'Wireless Mouse',      29.99,  'Electronics'),
    (2, 'Mechanical Keyboard', 149.99, 'Electronics'),
    (3, 'Standing Desk',       549.00, 'Furniture'),
    (4, 'Monitor Arm',         89.50,  'Furniture'),
    (5, 'USB-C Hub',           45.00,  'Electronics')
""")
print("✅ 5 products inserted")

rows
----
5   
✅ 5 products inserted


In [5]:
print("📋 Original schema and data:")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

📋 Original schema and data:

product_id | name                | price  | category   
-----------+---------------------+--------+------------
1          | Wireless Mouse      | 29.99  | Electronics
1          | Wireless Mouse      | 29.99  | Electronics
2          | Mechanical Keyboard | 149.99 | Electronics
2          | Mechanical Keyboard | 149.99 | Electronics
3          | Standing Desk       | 549.0  | Furniture  
3          | Standing Desk       | 549.0  | Furniture  
4          | Monitor Arm         | 89.5   | Furniture  
4          | Monitor Arm         | 89.5   | Furniture  
5          | USB-C Hub           | 45.0   | Electronics
5          | USB-C Hub           | 45.0   | Electronics


## 1.2 — Add a New Column

Business asks: _"We need to track product weight for shipping calculations."_

**In Hive:** You'd need to rewrite every Parquet file to add the column, or add it only at the end and hope positional reads don't break.

**In Iceberg:** One `ALTER TABLE` — instant, metadata-only. Existing rows return `NULL` for the new column.

In [6]:
run_query("ALTER TABLE iceberg.bronze.products ADD COLUMN weight_kg DOUBLE")
print("✅ Column 'weight_kg' added — zero files rewritten!")

✅ Column 'weight_kg' added — zero files rewritten!


In [7]:
print("📋 After ADD COLUMN — existing rows have NULL for weight_kg:")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

📋 After ADD COLUMN — existing rows have NULL for weight_kg:

product_id | name                | price  | category    | weight_kg
-----------+---------------------+--------+-------------+----------
1          | Wireless Mouse      | 29.99  | Electronics | None     
1          | Wireless Mouse      | 29.99  | Electronics | None     
2          | Mechanical Keyboard | 149.99 | Electronics | None     
2          | Mechanical Keyboard | 149.99 | Electronics | None     
3          | Standing Desk       | 549.0  | Furniture   | None     
3          | Standing Desk       | 549.0  | Furniture   | None     
4          | Monitor Arm         | 89.5   | Furniture   | None     
4          | Monitor Arm         | 89.5   | Furniture   | None     
5          | USB-C Hub           | 45.0   | Electronics | None     
5          | USB-C Hub           | 45.0   | Electronics | None     


New inserts can populate the new column immediately:

In [8]:
run_query("""
INSERT INTO iceberg.bronze.products VALUES
    (6, 'Laptop Stand', 79.99, 'Furniture', 2.3)
""")
print("✅ New product inserted with weight_kg = 2.3")

rows
----
1   
✅ New product inserted with weight_kg = 2.3


In [9]:
print("📋 Mixed data — old rows (NULL weight) + new row (with weight):")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

📋 Mixed data — old rows (NULL weight) + new row (with weight):

product_id | name                | price  | category    | weight_kg
-----------+---------------------+--------+-------------+----------
1          | Wireless Mouse      | 29.99  | Electronics | None     
1          | Wireless Mouse      | 29.99  | Electronics | None     
2          | Mechanical Keyboard | 149.99 | Electronics | None     
2          | Mechanical Keyboard | 149.99 | Electronics | None     
3          | Standing Desk       | 549.0  | Furniture   | None     
3          | Standing Desk       | 549.0  | Furniture   | None     
4          | Monitor Arm         | 89.5   | Furniture   | None     
4          | Monitor Arm         | 89.5   | Furniture   | None     
5          | USB-C Hub           | 45.0   | Electronics | None     
5          | USB-C Hub           | 45.0   | Electronics | None     
6          | Laptop Stand        | 79.99  | Furniture   | 2.3      


The old Parquet files were **never touched**. Iceberg reads the old files (which lack the `weight_kg` column) and fills in `NULL` automatically. The new file has the full schema. Both coexist seamlessly.

## 1.3 — Rename a Column

Business asks: _"Rename `category` to `department` to match our new taxonomy."_

**In Hive:** Impossible without breaking queries. Hive reads columns by **position**, not by name — renaming a column in the metastore would cause it to read the wrong data from existing files.

**In Iceberg:** Columns are tracked by **unique field IDs** embedded in the metadata. Renaming just updates the name → ID mapping. All existing data files remain valid.

In [10]:
run_query("ALTER TABLE iceberg.bronze.products RENAME COLUMN category TO department")
print("✅ Column 'category' → 'department' — zero files rewritten!")

✅ Column 'category' → 'department' — zero files rewritten!


In [11]:
print("📋 After RENAME COLUMN — 'category' is now 'department':")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

📋 After RENAME COLUMN — 'category' is now 'department':

product_id | name                | price  | department  | weight_kg
-----------+---------------------+--------+-------------+----------
1          | Wireless Mouse      | 29.99  | Electronics | None     
1          | Wireless Mouse      | 29.99  | Electronics | None     
2          | Mechanical Keyboard | 149.99 | Electronics | None     
2          | Mechanical Keyboard | 149.99 | Electronics | None     
3          | Standing Desk       | 549.0  | Furniture   | None     
3          | Standing Desk       | 549.0  | Furniture   | None     
4          | Monitor Arm         | 89.5   | Furniture   | None     
4          | Monitor Arm         | 89.5   | Furniture   | None     
5          | USB-C Hub           | 45.0   | Electronics | None     
5          | USB-C Hub           | 45.0   | Electronics | None     
6          | Laptop Stand        | 79.99  | Furniture   | 2.3      


### 🔍 Verify: Zero Data Files Were Touched

Let's confirm that the schema changes were metadata-only by inspecting the Iceberg snapshot history. We should see:
- **2 `append` snapshots** from the two INSERTs
- **No extra snapshots** from the ADD COLUMN or RENAME COLUMN — those didn't create new snapshots because no data was written.

In [12]:
print("📸 Snapshot history — only INSERTs created snapshots, schema changes did NOT:")
print()
run_query("""
SELECT committed_at, snapshot_id, operation
FROM iceberg.bronze."products$snapshots"
ORDER BY committed_at
""");

📸 Snapshot history — only INSERTs created snapshots, schema changes did NOT:

committed_at                     | snapshot_id         | operation
---------------------------------+---------------------+----------
2026-03-07 16:18:04.851000+00:00 | 6168253586156357558 | append   
2026-03-07 16:18:14.551000+00:00 | 540271601371068688  | append   
2026-03-07 16:18:37.281000+00:00 | 3973603003629479013 | append   
2026-03-07 16:19:09.797000+00:00 | 6156827495540134045 | append   


---
# Part 2 — Partition Evolution (Hidden Partitioning)

In Hive, partitioning is **baked into the directory structure** (`/year=2026/month=02/day=15/`). Changing the partition scheme means:
1. Create a brand-new table with the new partitioning
2. Copy (rewrite) all historical data
3. Migrate all downstream queries
4. Drop the old table

In Iceberg, partition specs are **tracked in metadata** and completely hidden from queries. You can evolve the partitioning at any time — old data keeps its original layout, new data uses the new scheme, and both are queried transparently.

```
                  Partition Evolution Timeline
──────────────────────────────────────────────────────────────
  Old data (spec v0)              New data (spec v1)
  Partitioned by MONTH            Partitioned by DAY
  ┌─────────┐                     ┌──────────────┐
  │ 2026-01 │                     │ 2026-03-01   │
  │ 2026-02 │  ── evolve ──▶      │ 2026-03-02   │
  │ 2026-03 │                     │ 2026-03-03   │
  └─────────┘                     └──────────────┘
        Both coexist — Iceberg handles it transparently
```

## 2.1 — Create an Event Table Partitioned by MONTH

In [13]:
run_query("DROP TABLE IF EXISTS iceberg.bronze.events")

run_query("""
CREATE TABLE iceberg.bronze.events (
    event_id   BIGINT,
    user_id    BIGINT,
    event_type VARCHAR,
    event_ts   TIMESTAMP(6) WITH TIME ZONE
) WITH (
    format = 'PARQUET',
    partitioning = ARRAY['month(event_ts)']
)
""")
print("✅ Table 'events' created — partitioned by MONTH(event_ts)")

✅ Table 'events' created — partitioned by MONTH(event_ts)


### 📥 Load Historical Data (Jan–Feb 2026)

Simulating a few months of event data. Iceberg automatically routes each row to the correct monthly partition — no explicit partition column needed.

In [14]:
run_query("""
INSERT INTO iceberg.bronze.events VALUES
    (1,  101, 'page_view',   TIMESTAMP '2026-01-05 10:00:00.000000 UTC'),
    (2,  102, 'click',       TIMESTAMP '2026-01-12 14:30:00.000000 UTC'),
    (3,  101, 'purchase',    TIMESTAMP '2026-01-20 09:15:00.000000 UTC'),
    (4,  103, 'page_view',   TIMESTAMP '2026-02-03 11:00:00.000000 UTC'),
    (5,  101, 'click',       TIMESTAMP '2026-02-14 16:45:00.000000 UTC'),
    (6,  104, 'signup',      TIMESTAMP '2026-02-28 08:30:00.000000 UTC'),
    (7,  102, 'page_view',   TIMESTAMP '2026-01-08 12:00:00.000000 UTC'),
    (8,  105, 'purchase',    TIMESTAMP '2026-02-10 15:20:00.000000 UTC'),
    (9,  103, 'click',       TIMESTAMP '2026-01-25 13:10:00.000000 UTC'),
    (10, 104, 'page_view',   TIMESTAMP '2026-02-18 10:05:00.000000 UTC')
""")
print("✅ 10 events inserted across Jan-Feb 2026")

rows
----
10  
✅ 10 events inserted across Jan-Feb 2026


In [15]:
print("📋 Events data:")
print()
run_query("SELECT * FROM iceberg.bronze.events ORDER BY event_ts");

📋 Events data:

event_id | user_id | event_type | event_ts                 
---------+---------+------------+--------------------------
1        | 101     | page_view  | 2026-01-05 10:00:00+00:00
7        | 102     | page_view  | 2026-01-08 12:00:00+00:00
2        | 102     | click      | 2026-01-12 14:30:00+00:00
3        | 101     | purchase   | 2026-01-20 09:15:00+00:00
9        | 103     | click      | 2026-01-25 13:10:00+00:00
4        | 103     | page_view  | 2026-02-03 11:00:00+00:00
8        | 105     | purchase   | 2026-02-10 15:20:00+00:00
5        | 101     | click      | 2026-02-14 16:45:00+00:00
10       | 104     | page_view  | 2026-02-18 10:05:00+00:00
6        | 104     | signup     | 2026-02-28 08:30:00+00:00


### 🔍 Inspect the Physical Partitioning

The `$partitions` metadata table shows how data is physically organized. Currently, data is grouped into monthly buckets.

In [16]:
print("📁 Partition layout (MONTH granularity):")
print()
run_query("""
SELECT
    partition.event_ts_month AS partition_value,
    record_count,
    file_count
FROM iceberg.bronze."events$partitions"
""");

📁 Partition layout (MONTH granularity):

partition_value | record_count | file_count
----------------+--------------+-----------
672             | 5            | 1         
673             | 5            | 1         


## 2.2 — Evolve Partition Spec: MONTH → DAY

The business grows, data volume increases. Monthly partitions are too coarse — queries that filter on a single day still scan the entire month.

**In Hive:** This is a nightmare. You'd need to create a new table, `INSERT OVERWRITE` all historical data with the new partitioning, update all downstream pipelines, and drop the old table. For a multi-TB table, this could take hours.

**In Iceberg:** One DDL statement. Existing data **stays in monthly partitions**. Only new writes use daily partitions. Iceberg's query planner handles both transparently.

In [17]:
run_query("""
ALTER TABLE iceberg.bronze.events
SET PROPERTIES partitioning = ARRAY['day(event_ts)']
""")
print("✅ Partition spec evolved: MONTH(event_ts) → DAY(event_ts)")
print("   Old data is untouched. New writes will use daily partitions.")

✅ Partition spec evolved: MONTH(event_ts) → DAY(event_ts)
   Old data is untouched. New writes will use daily partitions.


### 📥 Insert New Data (Mar 2026)

These new events will be written into **daily** partitions — completely transparent to the INSERT.

In [18]:
run_query("""
INSERT INTO iceberg.bronze.events VALUES
    (11, 101, 'page_view', TIMESTAMP '2026-03-01 09:00:00.000000 UTC'),
    (12, 102, 'click',     TIMESTAMP '2026-03-01 14:20:00.000000 UTC'),
    (13, 103, 'purchase',  TIMESTAMP '2026-03-02 11:30:00.000000 UTC'),
    (14, 105, 'signup',    TIMESTAMP '2026-03-02 16:00:00.000000 UTC'),
    (15, 101, 'click',     TIMESTAMP '2026-03-03 08:45:00.000000 UTC'),
    (16, 104, 'page_view', TIMESTAMP '2026-03-03 13:15:00.000000 UTC'),
    (17, 102, 'purchase',  TIMESTAMP '2026-03-04 10:00:00.000000 UTC'),
    (18, 106, 'signup',    TIMESTAMP '2026-03-04 15:30:00.000000 UTC')
""")
print("✅ 8 new events inserted (Mar 1-4) — using DAY partitions")

rows
----
8   
✅ 8 new events inserted (Mar 1-4) — using DAY partitions


### 🔍 Inspect the Mixed Partition Layout

Now the table has **two partition specs coexisting**:
- Old data → monthly partitions (spec v0)
- New data → daily partitions (spec v1)

We can see this by looking at the data files and their partitions:

In [19]:
print("📁 Data files — showing mixed monthly + daily partitions:")
print()
run_query("""
SELECT
    partition,
    record_count,
    file_size_in_bytes
FROM iceberg.bronze."events$files"
""");

📁 Data files — showing mixed monthly + daily partitions:

partition                                                       | record_count | file_size_in_bytes
----------------------------------------------------------------+--------------+-------------------
(event_ts_month: None, event_ts_day: datetime.date(2026, 3, 1)) | 2            | 703               
(event_ts_month: None, event_ts_day: datetime.date(2026, 3, 2)) | 2            | 703               
(event_ts_month: None, event_ts_day: datetime.date(2026, 3, 3)) | 2            | 703               
(event_ts_month: None, event_ts_day: datetime.date(2026, 3, 4)) | 2            | 703               
(event_ts_month: 672, event_ts_day: None)                       | 5            | 827               
(event_ts_month: 673, event_ts_day: None)                       | 5            | 836               


### ✅ Queries Work Transparently Across Both Specs

The key insight: **queries don't know or care about the partition layout**. Iceberg's planner reads from both monthly and daily partitions seamlessly.

In [20]:
print("📋 All events across ALL partition specs — seamless query:")
print()
run_query("SELECT * FROM iceberg.bronze.events ORDER BY event_ts");

📋 All events across ALL partition specs — seamless query:

event_id | user_id | event_type | event_ts                 
---------+---------+------------+--------------------------
1        | 101     | page_view  | 2026-01-05 10:00:00+00:00
7        | 102     | page_view  | 2026-01-08 12:00:00+00:00
2        | 102     | click      | 2026-01-12 14:30:00+00:00
3        | 101     | purchase   | 2026-01-20 09:15:00+00:00
9        | 103     | click      | 2026-01-25 13:10:00+00:00
4        | 103     | page_view  | 2026-02-03 11:00:00+00:00
8        | 105     | purchase   | 2026-02-10 15:20:00+00:00
5        | 101     | click      | 2026-02-14 16:45:00+00:00
10       | 104     | page_view  | 2026-02-18 10:05:00+00:00
6        | 104     | signup     | 2026-02-28 08:30:00+00:00
11       | 101     | page_view  | 2026-03-01 09:00:00+00:00
12       | 102     | click      | 2026-03-01 14:20:00+00:00
13       | 103     | purchase   | 2026-03-02 11:30:00+00:00
14       | 105     | signup     | 2026-03

In [21]:
print("📋 Filtered query — spans both old (monthly) and new (daily) partitions:")
print()
run_query("""
SELECT event_id, user_id, event_type, event_ts
FROM iceberg.bronze.events
WHERE event_ts >= TIMESTAMP '2026-02-01 00:00:00.000000 UTC'
  AND event_ts <  TIMESTAMP '2026-03-03 00:00:00.000000 UTC'
ORDER BY event_ts
""");

📋 Filtered query — spans both old (monthly) and new (daily) partitions:

event_id | user_id | event_type | event_ts                 
---------+---------+------------+--------------------------
4        | 103     | page_view  | 2026-02-03 11:00:00+00:00
8        | 105     | purchase   | 2026-02-10 15:20:00+00:00
5        | 101     | click      | 2026-02-14 16:45:00+00:00
10       | 104     | page_view  | 2026-02-18 10:05:00+00:00
6        | 104     | signup     | 2026-02-28 08:30:00+00:00
11       | 101     | page_view  | 2026-03-01 09:00:00+00:00
12       | 102     | click      | 2026-03-01 14:20:00+00:00
13       | 103     | purchase   | 2026-03-02 11:30:00+00:00
14       | 105     | signup     | 2026-03-02 16:00:00+00:00


In [22]:
print("📊 Events per day — aggregation works across both partition specs:")
print()
run_query("""
SELECT
    CAST(event_ts AS DATE) AS event_date,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_id) AS unique_users
FROM iceberg.bronze.events
GROUP BY CAST(event_ts AS DATE)
ORDER BY event_date
""");

📊 Events per day — aggregation works across both partition specs:

event_date | event_count | unique_users
-----------+-------------+-------------
2026-01-05 | 1           | 1           
2026-01-08 | 1           | 1           
2026-01-12 | 1           | 1           
2026-01-20 | 1           | 1           
2026-01-25 | 1           | 1           
2026-02-03 | 1           | 1           
2026-02-10 | 1           | 1           
2026-02-14 | 1           | 1           
2026-02-18 | 1           | 1           
2026-02-28 | 1           | 1           
2026-03-01 | 2           | 2           
2026-03-02 | 2           | 2           
2026-03-03 | 2           | 2           
2026-03-04 | 2           | 2           


---
## 📊 Summary

| Operation | SQL | Data Rewrite? | Downtime? |
|-----------|-----|:---:|:---:|
| **Add column** | `ALTER TABLE ... ADD COLUMN` | ❌ None | ❌ None |
| **Rename column** | `ALTER TABLE ... RENAME COLUMN` | ❌ None | ❌ None |
| **Evolve partitioning** | `ALTER TABLE ... SET PROPERTIES partitioning` | ❌ None | ❌ None |

All three operations are **metadata-only** — they complete in milliseconds regardless of whether your table has 6 rows or 6 billion. This is the fundamental advantage of Iceberg's approach of tracking schemas and partitions in table metadata rather than in the physical file layout.

---
## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to drop all tables:
# run_query("DROP TABLE IF EXISTS iceberg.bronze.products")
# run_query("DROP TABLE IF EXISTS iceberg.bronze.events")
# print("🗑️ All tables dropped")